Скачивание необходимых библиотек

In [ ]:
!pip install rdkit -qq

In [ ]:
!pip install optuna -qq

In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git

fatal: destination path 'rapidsai-csp-utils' already exists and is not an empty directory.


In [ ]:
!bash rapidsai-csp-utils/colab/rapids-colab.sh

PLEASE READ FOR 21.06
********************************************************************************************************
Another release, another script change.  We had to revise the script, which now:
1. Does a more comprehensive install
2. Includes BlazingSQL
3. is far easier for everyone to understand and maintain

The script will require you to add these 5 cells to your notebook.  We have also created a new startup template: 
https://colab.research.google.com/drive/1TAAi_szMfWqRfHVfjGSqnGVLr_ztzUM9?usp=sharing

CHANGES T
CELL 1:
    # This get the RAPIDS-Colab install files and test check your GPU.  Run cells 1 and 2 only.
    # Please read the output of this cell.  If your Colab Instance is not RAPIDS compatible, it will warn you and give you remediation steps.
    !git clone https://github.com/rapidsai/rapidsai-csp-utils.git
    !python rapidsai-csp-utils/colab/env-check.py

CELL 2:
    # This will update the Colab environment and restart the kernel.
    !bash rapidsai-csp-

Импортирование библиотек

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from cuml.ensemble import RandomForestRegressor
from rdkit.Chem import PandasTools
from rdkit import DataStructs
import optuna

Чтение файла и удаление нулевых элементов

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/BigSolDB/BigSolDBv2.0.csv')
df = df.dropna()

Просмотр данных

In [ ]:
df.columns

Index(['SMILES_Solute', 'Temperature_K', 'Solvent', 'SMILES_Solvent',
       'Solubility(mole_fraction)', 'Solubility(mol/L)', 'LogS(mol/L)',
       'Compound_Name', 'CAS', 'PubChem_CID', 'FDA_Approved', 'Source'],
      dtype='object')

In [ ]:
df['Solvent'].unique()

array(['ethanol', 'methanol', 'isopropanol', 'water', 'ethyl acetate',
       'n-propanol', 'acetone', 'n-butanol', 'acetonitrile', 'DMF',
       'toluene', 'isobutanol', '1,4-dioxane', 'methyl acetate', 'THF',
       '2-butanone', 'n-pentanol', 'sec-butanol', 'n-hexane',
       'ethylene glycol', 'NMP', 'cyclohexane', 'DMSO', 'n-butyl acetate',
       'n-octanol', 'chloroform', 'n-propyl acetate', 'acetic acid',
       'dichloromethane', 'cyclohexanone', 'propylene glycol',
       'isopropyl acetate', 'DMAc', '2-ethoxyethanol', 'isopentanol',
       'n-heptane', 'ethyl formate', 'benzene', '1,2-dichloroethane',
       'n-hexanol', '2-methoxyethanol', 'isobutyl acetate',
       'tetrachloromethane', 'n-pentyl acetate', 'transcutol',
       'n-heptanol', 'ethylbenzene', 'MIBK', '2-propoxyethanol',
       'tert-butanol', 'MTBE', '2-butoxyethanol', 'propionic acid',
       'o-xylene', 'formic acid', 'diethyl ether', 'm-xylene', 'p-xylene',
       'chlorobenzene', 'dimethyl carbonate', 'n-

Создание копии исходного датасета для обработки признаков

In [ ]:
df_processed = df.copy()

Нормализация представлений молекул

In [ ]:
PandasTools.AddMoleculeColumnToFrame(
    df_processed,
    'SMILES_Solute',
    'Mol_Solute')
PandasTools.AddMoleculeColumnToFrame(
    df_processed,
    'SMILES_Solvent',
    'Mol_Solvent')
df_processed.head()

,SMILES_Solute,Temperature_K,Solvent,SMILES_Solvent,Solubility(mole_fraction),Solubility(mol/L),LogS(mol/L),Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,Mol_Solute,Mol_Solvent
0,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,311.25,ethanol,CCO,0.0006,0.010083,-1.996419,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b2024430>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027d80>
1,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,314.65,ethanol,CCO,0.0012,0.020100,-1.696799,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b2024040>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027680>
2,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,319.15,ethanol,CCO,0.0020,0.033356,-1.476824,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b2026ce0>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027b50>
3,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,322.15,ethanol,CCO,0.0050,0.083356,-1.079064,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b20251c0>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027bc0>
4,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,324.15,ethanol,CCO,0.0139,0.233286,-0.632111,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b2026c70>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027d10>


Получение моргановских отпечатков для каждой молекулы

In [ ]:
from rdkit.Chem import AllChem

def morgan_fp(mol):
  morgan = AllChem.GetMorganGenerator(radius=2, fpSize=512)
  return morgan.GetFingerprint(mol)

In [ ]:
df_processed['Morgan_Solute'] = df_processed['Mol_Solute'].apply(morgan_fp)
df_processed['Morgan_Solvent'] = df_processed['Mol_Solvent'].apply(morgan_fp)
df_processed.head()

,SMILES_Solute,Temperature_K,Solvent,SMILES_Solvent,Solubility(mole_fraction),Solubility(mol/L),LogS(mol/L),Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,Mol_Solute,Mol_Solvent,Morgan_Solute,Morgan_Solvent
0,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,311.25,ethanol,CCO,0.0006,0.010083,-1.996419,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b2024430>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027d80>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,314.65,ethanol,CCO,0.0012,0.020100,-1.696799,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b2024040>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027680>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,319.15,ethanol,CCO,0.0020,0.033356,-1.476824,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b2026ce0>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027b50>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,322.15,ethanol,CCO,0.0050,0.083356,-1.079064,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b20251c0>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027bc0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,324.15,ethanol,CCO,0.0139,0.233286,-0.632111,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x78b7b2026c70>,<rdkit.Chem.rdchem.Mol object at 0x78b7b2027d10>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
df_processed.columns

Index(['SMILES_Solute', 'Temperature_K', 'Solvent', 'SMILES_Solvent',
       'Solubility(mole_fraction)', 'Solubility(mol/L)', 'LogS(mol/L)',
       'Compound_Name', 'CAS', 'PubChem_CID', 'FDA_Approved', 'Source',
       'Mol_Solute', 'Mol_Solvent', 'Morgan_Solute', 'Morgan_Solvent'],
      dtype='object')

Разделение выборки на обучающую и целевую функцию

In [ ]:
X = df_processed[['Temperature_K',
                  'Morgan_Solute',
                  'Morgan_Solvent']]
y = df_processed['LogS(mol/L)']

In [ ]:
X.head()

,Temperature_K,Morgan_Solute,Morgan_Solvent
0,311.25,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,314.65,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,319.15,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,322.15,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,324.15,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
y.head()

,LogS(mol/L)
0,-1.996419
1,-1.696799
2,-1.476824
3,-1.079064
4,-0.632111


In [ ]:
X['Morgan_Solute'] = X['Morgan_Solute'].tolist()
X['Morgan_Solvent'] = X['Morgan_Solvent'].tolist()

/tmp/ipython-input-1126101153.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Morgan_Solute'] = X['Morgan_Solute'].tolist()
/tmp/ipython-input-1126101153.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Morgan_Solvent'] = X['Morgan_Solvent'].tolist()


Приведение моргановских отпечатков типа bitvect к массиву numpy

In [ ]:
def bitvect_to_array(bitvect):
  arr = np.zeros((1,), dtype=int)
  DataStructs.ConvertToNumpyArray(bitvect, arr)
  return arr

In [ ]:
solute_fp = np.vstack(X['Morgan_Solute'].values)
solvent_fp = np.vstack(X['Morgan_Solvent'].values)

Вертикальное соединения массивов (по строкам)

Объединение данных для получения результирующей выборки

In [ ]:
X_base = X.drop(['Morgan_Solute', 'Morgan_Solvent'], axis=1)

solute_df = pd.DataFrame(solute_fp, index=X.index).add_prefix('SoluteFP_')
solvent_df = pd.DataFrame(solvent_fp, index=X.index).add_prefix('SolventFP_')

X_final = pd.concat([X_base, solute_df, solvent_df], axis=1)

In [ ]:
X_final.shape, y.shape

((93929, 1025), (93929,))

Разделение на трейн и тест для обучения модели

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y,
    test_size=0.2,
    random_state=42
)

Подбор оптимальных гиперпараметров для модели случайного леса

In [ ]:
def objective(trial):
  n_estimators = trial.suggest_int('n_estimators', 10, 200)
  max_depth = trial.suggest_int('max_depth', 2, 32)
  min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
  max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', 'auto'])

  model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  rmse = mean_squared_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)

  print(f"Trial {trial.number}: RMSE={rmse:.4f}, R2={r2:.4f}")

  return rmse

Обучение модели

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print(f"Best trial value (RMSE): {study.best_value}")
print(f"Best hyperparameters: {study.best_params}")

[I 2025-10-10 19:51:57,296] A new study created in memory with name: no-name-da8ae895-dc73-4755-a657-f3fc49a6d133
/usr/local/lib/python3.12/dist-packages/cuml/internals/api_decorators.py:216: FutureWarning: `max_features='auto'` has been deprecated in 24.06 and will be removed in 25.08. To keep the past behaviour and silence this warning, explicitly set `max_features=1.0`.
  ret = func(*args, **kwargs)
[I 2025-10-10 19:54:11,793] Trial 0 finished with value: 0.31486029874553617 and parameters: {'n_estimators': 83, 'max_depth': 25, 'min_samples_split': 4, 'min_samples_leaf': 10, 'max_features': 'auto'}. Best is trial 0 with value: 0.31486029874553617.


Trial 0: RMSE=0.3149, R2=0.7865


/usr/local/lib/python3.12/dist-packages/cuml/internals/api_decorators.py:216: FutureWarning: `max_features='auto'` has been deprecated in 24.06 and will be removed in 25.08. To keep the past behaviour and silence this warning, explicitly set `max_features=1.0`.
  ret = func(*args, **kwargs)
[I 2025-10-10 19:55:59,694] Trial 1 finished with value: 0.21153558255668264 and parameters: {'n_estimators': 62, 'max_depth': 27, 'min_samples_split': 11, 'min_samples_leaf': 3, 'max_features': 'auto'}. Best is trial 1 with value: 0.21153558255668264.


Trial 1: RMSE=0.2115, R2=0.8565


[I 2025-10-10 19:56:01,917] Trial 2 finished with value: 0.3737419943043532 and parameters: {'n_estimators': 24, 'max_depth': 26, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.21153558255668264.


...

Trial 27: RMSE=0.7295, R2=0.5053


[I 2025-10-10 20:32:39,978] Trial 28 finished with value: 0.681600065604192 and parameters: {'n_estimators': 76, 'max_depth': 29, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 12 with value: 0.16747726747507632.


Trial 28: RMSE=0.6816, R2=0.5378


[I 2025-10-10 20:32:46,437] Trial 29 finished with value: 0.47261848214624946 and parameters: {'n_estimators': 98, 'max_depth': 26, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 12 with value: 0.16747726747507632.


Trial 29: RMSE=0.4726, R2=0.6795
Best trial value (RMSE): 0.16747726747507632
Best hyperparameters: {'n_estimators': 77, 'max_depth': 31, 'min_samples_split': 12, 'min_samples_leaf': 4, 'max_features': 'auto'}


In [ ]:
rfr_model = RandomForestRegressor(
        n_estimators=study.best_params['n_estimators'],
        max_depth=study.best_params['max_depth'],
        min_samples_split=study.best_params['min_samples_split'],
        min_samples_leaf=study.best_params['min_samples_leaf'],
        max_features='auto',
        random_state=42
    )
rfr_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/cuml/internals/api_decorators.py:216: FutureWarning: `max_features='auto'` has been deprecated in 24.06 and will be removed in 25.08. To keep the past behaviour and silence this warning, explicitly set `max_features=1.0`.
  ret = func(*args, **kwargs)


RandomForestRegressor()

In [ ]:
from joblib import dump, load
from pathlib import Path

model_dir_path = '/content/drive/MyDrive/RFR_solubility'

dump(rfr_model, Path(model_dir_path) / 'rfr_for_solubility_pred.joblib')

['/content/drive/MyDrive/RFR_solubility/rfr_for_solubility_pred.joblib']